# 15b — Des spreads réalistes : calibrer comme un vrai desk

## Pourquoi les spreads du notebook 15 étaient énormes

Deux paramètres irréalistes gonflaient tout :

1. **Le coût de transaction à 1%.** Un desk qui couvre un sous-jacent liquide (indice, action large cap, future) paie l'ordre de **quelques points de base**, pas 1%. Et il rééquilibre souvent (quotidien, voire intraday), pas une fois par semaine.
2. **Le critère de risque « CVaR 95% = 0 ».** Ça exige qu'une transaction *isolée* soit rentable dans son propre pire 5%. Aucun desk ne price comme ça : il **mutualise le risque sur tout son livre**, donc le risque *marginal* d'une transaction de plus est faible, et il ne facture que l'espérance du coût plus une petite marge.

On modélise le second point avec une **aversion au risque** $\lambda \in [0,1]$ via la mesure

$$\rho_\lambda(L) = (1-\lambda)\,\mathbb{E}[L] + \lambda\,\text{CVaR}_\alpha(L).$$

$\lambda = 1$ redonne le CVaR plein (transaction isolée, ultra-prudent) ; $\lambda \to 0$ décrit un desk très diversifié qui ne charge presque que le coût espéré. Le prix vendeur devient $p_{\text{ask}} = e^{-rT}\rho_\lambda(\text{payoff} - \text{gains})$, et le spread $e^{-rT}[\rho_\lambda(L_{\text{ask}}) + \rho_\lambda(L_{\text{bid}})]$. Le point clé : $\rho_\lambda$ reste une **mesure de risque cohérente** (combinaison convexe de l'espérance et du CVaR, tous deux cohérents).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

S0,K,r,T,sigma = 100.,100.,0.02,1.0,0.20; Bar=130.
def d12(S,tau): d1=(np.log(S/K)+(r+0.5*sigma**2)*tau)/(sigma*np.sqrt(tau)); return d1,d1-sigma*np.sqrt(tau)
def bs_call(S,tau): d1,d2=d12(S,tau); return S*norm.cdf(d1)-K*np.exp(-r*tau)*norm.cdf(d2)
def bs_delta(S,tau): d1,_=d12(S,tau); return norm.cdf(d1)
def CVaR(L,a): return L[L>=np.quantile(L,a)].mean()
def rho(L,a,lam): return (1-lam)*L.mean()+lam*CVaR(L,a)
pBS=float(bs_call(S0,T)); alpha=0.95
print(f"prix BS du call = {pBS:.3f}")


## 1. Le vanille : carte du spread selon le coût et l'aversion

Pour le call, le delta-hedge est quasi optimal, donc on peut tout faire en numpy. On rééquilibre fréquemment (n=126, tous les deux jours), on balaie plusieurs niveaux de coût, et pour chaque on calcule le spread sous $\rho_\lambda$ pour toute une plage de $\lambda$ (l'aversion se recalcule sans re-simuler, elle ne fait que repondérer moyenne et queue).


In [ ]:
n=126; dt=T/n; times=np.linspace(0,T,n+1); rng=np.random.default_rng(0); m=60000
Z=rng.standard_normal((m,n)); S=S0*np.exp(np.concatenate([np.zeros((m,1)),np.cumsum((r-0.5*sigma**2)*dt+sigma*np.sqrt(dt)*Z,axis=1)],axis=1))
payoff=np.maximum(S[:,-1]-K,0.)
def gains(sign,cost):
    cash=np.zeros(m); pos=np.zeros(m)
    for k in range(n):
        tau=max(T-times[k],1e-4); tgt=sign*bs_delta(S[:,k],tau); tr=tgt-pos
        cash-=tr*S[:,k]+cost*np.abs(tr)*S[:,k]; pos+=tr; cash*=np.exp(r*dt)
    return cash+pos*S[:,-1]
costs=np.array([1,2,5,10,20,50])*1e-4          # de 1 a 50 bps
lams=np.linspace(0.02,1.0,25)
disc=np.exp(-r*T)
grid=np.zeros((len(costs),len(lams)))
for i,c in enumerate(costs):
    La=payoff-gains(+1,c); Lb=-gains(-1,c)-payoff
    for j,lam in enumerate(lams):
        grid[i,j]=disc*(rho(La,alpha,lam)+rho(Lb,alpha,lam))/pBS   # spread relatif
fig,ax=plt.subplots(figsize=(9,4.5))
im=ax.pcolormesh(lams,np.arange(len(costs)),grid,shading='auto',cmap='viridis')
ax.set_yticks(np.arange(len(costs))); ax.set_yticklabels([f"{int(c*1e4)} bps" for c in costs])
fig.colorbar(im,label='spread / prix'); ax.set_xlabel('aversion lambda'); ax.set_ylabel('cout de transaction')
ax.contour(lams,np.arange(len(costs)),grid,levels=[0.02,0.05,0.10],colors='white',linewidths=1)
ax.set_title('Call : spread relatif (lignes blanches = 2%, 5%, 10%)')
fig.tight_layout(); plt.show()
# quelques points de fonctionnement
for c,lam in [(0.0005,0.2),(0.0002,0.1),(0.0002,0.05)]:
    La=payoff-gains(+1,c); Lb=-gains(-1,c)-payoff
    s=disc*(rho(La,alpha,lam)+rho(Lb,alpha,lam))
    print(f"cout {int(c*1e4)} bps, lambda={lam}: spread={s:.3f} soit {s/pBS:.1%} du prix")


Un vanille liquide se cote autour de **quelques pour cent de la prime**, et la carte montre qu'on y arrive avec des paramètres de desk plausibles (quelques bps, aversion faible). Les deux leviers agissent différemment : le coût fixe surtout le **milieu** (dépense espérée de couverture), l'aversion $\lambda$ fixe surtout la **largeur** (marge de risque).


## 2. Les exotiques : pourquoi le deep hedger devient nécessaire

Au même point réaliste, le delta-hedge classique donne un spread crédible pour le vanille mais **reste énorme pour la barrière**, parce que son risque de gap (delta qui saute au seuil, désactivation) est **intrinsèque**, pas dû au coût. Baisser le coût ne le corrige pas.


In [ ]:
def uo(S,tau):
    S=np.asarray(S,float); H=Bar; srt=sigma*np.sqrt(tau); m_=(r-0.5*sigma**2)/sigma**2
    x1=np.log(S/K)/srt+(1+m_)*srt; x2=np.log(S/H)/srt+(1+m_)*srt
    y1=np.log(H**2/(S*K))/srt+(1+m_)*srt; y2=np.log(H/S)/srt+(1+m_)*srt
    A=S*norm.cdf(x1)-K*np.exp(-r*tau)*norm.cdf(x1-srt); B=S*norm.cdf(x2)-K*np.exp(-r*tau)*norm.cdf(x2-srt)
    C=S*(H/S)**(2*(m_+1))*norm.cdf(-y1)-K*np.exp(-r*tau)*(H/S)**(2*m_)*norm.cdf(-y1+srt)
    D=S*(H/S)**(2*(m_+1))*norm.cdf(-y2)-K*np.exp(-r*tau)*(H/S)**(2*m_)*norm.cdf(-y2+srt)
    return np.where(S>=H,0.,A-B+C-D)
def uo_delta(S,tau,h=0.5): return (uo(S+h,tau)-uo(S-h,tau))/(2*h)
alive=np.cumprod((S<Bar).astype(float),axis=1)
pay_bar=np.where(alive[:,-1]>0,np.maximum(S[:,-1]-K,0.),0.); RN_bar=float(uo(S0,T))
c,lam=0.0005,0.2
def gains_bar(sign):
    cash=np.zeros(m); pos=np.zeros(m)
    for k in range(n):
        tau=max(T-times[k],1e-4); tgt=sign*uo_delta(S[:,k],tau)*alive[:,k]; tr=tgt-pos
        cash-=tr*S[:,k]+c*np.abs(tr)*S[:,k]; pos+=tr; cash*=np.exp(r*dt)
    return cash+pos*S[:,-1]
sb=disc*(rho(pay_bar-gains_bar(1),alpha,lam)+rho(-gains_bar(-1)-pay_bar,alpha,lam))
print(f"barriere, classique, point realiste (5bps, lambda=0.2): spread = {sb:.3f} soit {sb/RN_bar:.0%} du prix")
print("=> le delta classique ne suffit pas : c'est le deep hedger (notebook 13) qui resserre ce spread,")
print("   car il apprend a couper la couverture pres du seuil, ce que le delta ne fait pas.")


## Ce qu'il faut retenir

- **Oui, on obtient des spreads réalistes** en calibrant comme un desk : coût en points de base, rééquilibrage fréquent, et surtout une aversion $\lambda$ faible qui traduit la mutualisation du risque sur le livre. Pour un vanille liquide, on retombe sur quelques pour cent de la prime.
- **Le spread se décompose proprement** : un milieu piloté par le coût de couverture espéré, et une demi-largeur pilotée par l'aversion $\lambda$. C'est exactement le vocabulaire d'un desk (coût de portage plus marge de risque).
- **Sur les exotiques, la couverture classique plafonne** : le risque de gap de la barrière est intrinsèque et garde le spread large même à coût nul. Le deep hedger, lui, apprend une politique qui réduit ce risque, donc il permet de coter l'exotique nettement plus serré. Le lien qualité de couverture vers compétitivité de la cote, déjà vu au notebook 15, devient ici quantitatif et réaliste.
- La conclusion d'ensemble tient en une phrase défendable en entretien : *un desk cote le coût de couverture plus une marge de risque calibrée sur son appétit ; mieux il couvre, plus la marge est fine, et c'est sur les exotiques mal couverts par les grecques que le deep hedging change vraiment la cote.*
